# 02_Narratives_Cleaning

In [1]:
# To Do: AI Use Disclaimer

In [2]:
import pandas as pd
import duckdb
import polars as pl

## Loading the Data - the 5 Products

In [3]:
# Load parquet into lazy dataframe and display the head
PARQUET_PATH = '../data/processed/consumer_banking_complaints.parquet'
df = pl.scan_parquet(PARQUET_PATH)

print(f"Total number of records: {df.select(pl.len()).collect()}")
df.head(5).collect()

Total number of records: shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 1133355 │
└─────────┘


Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
date,str,str,str,str,str,str,str,str,str,str,str,str,date,str,bool,str,i64
2019-12-26,"""Credit card or prepaid card""","""General-purpose credit card or…","""Advertising and marketing, inc…","""Confusing or misleading advert…",null,null,"""CAPITAL ONE FINANCIAL CORPORAT…","""CA""","""94025""",null,"""Consent not provided""","""Web""",2019-12-26,"""Closed with explanation""",true,"""N/A""",3477549
2019-12-20,"""Checking or savings account""","""Other banking product or servi…","""Managing an account""","""Funds not handled or disbursed…",null,"""Company has responded to the c…","""WELLS FARGO & COMPANY""","""FL""","""33064""",null,"""N/A""","""Referral""",2019-12-23,"""Closed with explanation""",true,"""N/A""",3475858
2019-11-18,"""Credit card or prepaid card""","""General-purpose credit card or…","""Problem with a purchase shown …","""Credit card company isn't reso…","""XXXX claimed they delivered a …",null,"""DISCOVER BANK""","""MA""","""021XX""",null,"""Consent provided""","""Web""",2019-11-18,"""Closed with explanation""",true,"""N/A""",3442136
2020-06-05,"""Checking or savings account""","""Checking account""","""Managing an account""","""Problem using a debit or ATM c…",null,"""Company has responded to the c…","""CITIBANK, N.A.""","""NY""","""10466""",null,"""Consent not provided""","""Web""",2020-06-05,"""Closed with explanation""",true,"""N/A""",3684669
2024-01-16,"""Credit card""","""General-purpose credit card or…","""Other features, terms, or prob…","""Add-on products and services""",null,"""Company has responded to the c…","""WELLS FARGO & COMPANY""","""TX""","""76179""",null,"""Consent not provided""","""Web""",2024-01-16,"""Closed with monetary relief""",true,"""N/A""",8161600


In [4]:
# # Drop null/blank narratives
#
# narratives_df = (
#     df
#     .filter(
#         pl.col('Consumer complaint narrative').is_not_null()
#         & (pl.col('Consumer complaint narrative').str.strip_chars() != '')
#     )
#     .collect()
#     .to_pandas()
# )

In [5]:
# Total row count
row_count = (
    df
    .select(pl.len())
    .collect()
    .item()
)

print(row_count)

1133355


In [6]:
# Get non-null counts for each column
non_null_counts = (
    df
    .select(pl.all().count())
    .collect()
)

list(non_null_counts.to_dicts()[0].items())

[('Date received', 1133355),
 ('Product', 1133355),
 ('Sub-product', 1105056),
 ('Issue', 1133350),
 ('Sub-issue', 869126),
 ('Consumer complaint narrative', 541208),
 ('Company public response', 562549),
 ('Company', 1133355),
 ('State', 1114289),
 ('ZIP code', 1119483),
 ('Tags', 201520),
 ('Consumer consent provided?', 1083733),
 ('Submitted via', 1133355),
 ('Date sent to company', 1133355),
 ('Company response to consumer', 1133354),
 ('Timely response?', 1133355),
 ('Consumer disputed?', 1133355),
 ('Complaint ID', 1133355)]

TO DO: Discuss dupes with Noah

In [7]:
# Drop exact duplicates on narrative AND company response to consumer
# df_model = df.unique(
#     subset=[
#         'Consumer complaint narrative',
#         'Company response to consumer'
#     ],
#     keep='first'
# )

In [8]:
# Replace blank/null narratives with '[No Narrative]'
df_cleaned = (
    df
    .with_columns(
        pl.when(
            pl.col('Consumer complaint narrative').is_null() |
            (pl.col('Consumer complaint narrative').str.strip_chars() == '')
        )
        .then(pl.lit('[No Narrative]'))
        .otherwise(pl.col('Consumer complaint narrative'))
        .alias('Consumer complaint narrative')
    )
)

In [9]:
# def drop_null_rows(
#     lf: pl.LazyFrame,
#     columns: list[str]
# ) -> pl.LazyFrame:
#     """
#     Drop rows from a Polars LazyFrame with null values for any of the specified columns.
#     Prints records dropped and remaining records after each column-level drop.
#     """
#     lf_cleaned = lf
#
#     starting_count = (
#         lf_cleaned
#         .select(pl.len())
#         .collect()
#         .item()
#     )
#
#     print(f'Starting records: {starting_count:,}')
#
#     for col in columns:
#         before_count = (
#             lf_cleaned
#             .select(pl.len())
#             .collect()
#             .item()
#         )
#
#         lf_cleaned = lf_cleaned.filter(
#             pl.col(col).is_not_null()
#         )
#
#         after_count = (
#             lf_cleaned
#             .select(pl.len())
#             .collect()
#             .item()
#         )
#
#         records_dropped = before_count - after_count
#
#         print(f'\nColumn: {col}')
#         print(f'Records dropped: {records_dropped:,}')
#         print(f'Records remaining: {after_count:,}')
#
#     final_count = (
#         lf_cleaned
#         .select(pl.len())
#         .collect()
#         .item()
#     )
#
#     print(f'\nTotal records after all drops are complete: {final_count:,}')
#
#     return lf_cleaned

In [10]:
# Drop nulls for Sub-product and Sub-issue
#
# columns_to_check = [
#     'Sub-product',
#     'Sub-issue',
# ]
#
# df = drop_null_rows(df, columns_to_check)

In [11]:
# Count records containing redactions
records_with_redactions = (
    df_cleaned
    .filter(
        pl.col('Consumer complaint narrative')
        .fill_null('')
        .str.contains(r'XX/XX/XXXX|X{2,}')
    )
    .select(pl.len())
    .collect()
    .item()
)

print(f'Records with redaction patterns: {records_with_redactions:,}')

Records with redaction patterns: 458,607


In [12]:
# Replace CFPB redactions and normalize whitespace for all narratives.
# Creates new column with the cleaned version.
records_replaced = (
    df_cleaned
    .filter(
        pl.col('Consumer complaint narrative')
        .fill_null('')
        .str.contains(r'XX/XX/XXXX|X{2,}')
    )
    .select(pl.len())
    .collect()
    .item()
)

df_cleaned = df_cleaned.with_columns(
    pl.col('Consumer complaint narrative')
    .fill_null('')
    .str.replace_all(r'XX/XX/XXXX', ' REDACTED_DATE ')
    .str.replace_all(r'X{2,}', ' REDACTED ')
    # Normalize whitespace introduced by replacements and existing whitespace
    .str.replace_all(r'\s+', ' ')
    .str.strip_chars()
    .alias('cleaned_consumer_narrative')
)

print(f'Records with replacements: {records_replaced:,}')

Records with replacements: 458,607


In [13]:
# Check for encoding issues
encoding_issues = (
    df_cleaned
    .filter(
        pl.col('cleaned_consumer_narrative')
        .str.contains('\uFFFD', literal=True)
    )
    .select('cleaned_consumer_narrative')
    .limit(20)
    .collect()
)

if len(encoding_issues) == 0:
    print("No encoding issues found.")

No encoding issues found.


In [14]:
# Sanity check - both narrative columns have the 'no narrative' placeholder

samples = (
    df_cleaned
    .filter(
        pl.col('Consumer complaint narrative')
        .fill_null('')
        .str.contains(r'\[No Narrative\]')
        |
        pl.col('cleaned_consumer_narrative')
        .fill_null('')
        .str.contains(r'\[No Narrative\]')
    )
    .select([
        'Consumer complaint narrative',
        'cleaned_consumer_narrative'
    ])
    .limit(5)
    .collect()
)

samples

Consumer complaint narrative,cleaned_consumer_narrative
str,str
"""[No Narrative]""","""[No Narrative]"""
"""[No Narrative]""","""[No Narrative]"""
"""[No Narrative]""","""[No Narrative]"""
"""[No Narrative]""","""[No Narrative]"""
"""[No Narrative]""","""[No Narrative]"""


In [15]:
# Sanity check - total record count and non-null narratives in cleaned lazyframe

validation_counts = (
    df_cleaned
    .select(
        pl.len().alias('total_records'),

        (
            pl.col('cleaned_consumer_narrative')
            .fill_null('')
            .ne('[No Narrative]')
        )
        .sum()
        .alias('records_with_narrative')
    )
    .collect()
)

print(validation_counts)

shape: (1, 2)
┌───────────────┬────────────────────────┐
│ total_records ┆ records_with_narrative │
│ ---           ┆ ---                    │
│ u32           ┆ u32                    │
╞═══════════════╪════════════════════════╡
│ 1133355       ┆ 541208                 │
└───────────────┴────────────────────────┘


In [16]:
df_cleaned.head().collect()

Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_consumer_narrative
date,str,str,str,str,str,str,str,str,str,str,str,str,date,str,bool,str,i64,str
2019-12-26,"""Credit card or prepaid card""","""General-purpose credit card or…","""Advertising and marketing, inc…","""Confusing or misleading advert…","""[No Narrative]""",null,"""CAPITAL ONE FINANCIAL CORPORAT…","""CA""","""94025""",null,"""Consent not provided""","""Web""",2019-12-26,"""Closed with explanation""",true,"""N/A""",3477549,"""[No Narrative]"""
2019-12-20,"""Checking or savings account""","""Other banking product or servi…","""Managing an account""","""Funds not handled or disbursed…","""[No Narrative]""","""Company has responded to the c…","""WELLS FARGO & COMPANY""","""FL""","""33064""",null,"""N/A""","""Referral""",2019-12-23,"""Closed with explanation""",true,"""N/A""",3475858,"""[No Narrative]"""
2019-11-18,"""Credit card or prepaid card""","""General-purpose credit card or…","""Problem with a purchase shown …","""Credit card company isn't reso…","""XXXX claimed they delivered a …",null,"""DISCOVER BANK""","""MA""","""021XX""",null,"""Consent provided""","""Web""",2019-11-18,"""Closed with explanation""",true,"""N/A""",3442136,"""REDACTED claimed they delivere…"
2020-06-05,"""Checking or savings account""","""Checking account""","""Managing an account""","""Problem using a debit or ATM c…","""[No Narrative]""","""Company has responded to the c…","""CITIBANK, N.A.""","""NY""","""10466""",null,"""Consent not provided""","""Web""",2020-06-05,"""Closed with explanation""",true,"""N/A""",3684669,"""[No Narrative]"""
2024-01-16,"""Credit card""","""General-purpose credit card or…","""Other features, terms, or prob…","""Add-on products and services""","""[No Narrative]""","""Company has responded to the c…","""WELLS FARGO & COMPANY""","""TX""","""76179""",null,"""Consent not provided""","""Web""",2024-01-16,"""Closed with monetary relief""",true,"""N/A""",8161600,"""[No Narrative]"""


In [17]:
# Save to parquet file
OUTPUT_PATH = '../data/processed/cleaned_complaints.parquet'

df_cleaned.collect().write_parquet(OUTPUT_PATH)